In [1]:
import warnings
warnings.filterwarnings('ignore')

# Importing necessary libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau
import plotly.graph_objects as go
import plotly.express as px

In [2]:
# Paths to the dataset
dataset_path = "../../data/animals"
labels_file = "../../data/name of the animals.txt"

In [3]:
# Reading labels
with open(labels_file, 'r') as f:
    animal_names = f.read().split('\n')

In [4]:
animal_names[:10]

['antelope',
 'badger',
 'bat',
 'bear',
 'bee',
 'beetle',
 'bison',
 'boar',
 'butterfly',
 'cat']

In [5]:
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.resize(image, (180, 180))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = image / 255.0
    return image

In [6]:
data = []
labels = []

for animal in animal_names:
    animal_dir = os.path.join(dataset_path, animal)
    for img_name in os.listdir(animal_dir):
        img_path = os.path.join(animal_dir, img_name)
        data.append(preprocess_image(img_path))
        labels.append(animal)

data = np.array(data)
labels = np.array(labels)

In [7]:
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42)

In [9]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

In [23]:
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(180, 180, 3))
base_model.trainable = False  # Giai đoạn 1: Freeze toàn bộ

In [24]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(len(animal_names), activation='softmax')
])

In [25]:
# Compile and train (Giai đoạn 1)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

In [26]:
print("==== GIAI ĐOẠN 1: Huấn luyện head ====")
history1 = model.fit(datagen.flow(X_train, y_train, batch_size=32), 
                     epochs=10, 
                     validation_data=(X_test, y_test))

==== GIAI ĐOẠN 1: Huấn luyện head ====
Epoch 1/10
135/135 [==============================] - 21s 129ms/step - loss: 3.0488 - accuracy: 0.3465 - val_loss: 1.2358 - val_accuracy: 0.7009
Epoch 2/10
135/135 [==============================] - 14s 106ms/step - loss: 1.6050 - accuracy: 0.6037 - val_loss: 0.8740 - val_accuracy: 0.7676
Epoch 3/10
135/135 [==============================] - 14s 107ms/step - loss: 1.3627 - accuracy: 0.6530 - val_loss: 0.7881 - val_accuracy: 0.7796
Epoch 4/10
135/135 [==============================] - 14s 107ms/step - loss: 1.2080 - accuracy: 0.6794 - val_loss: 0.7888 - val_accuracy: 0.7759
Epoch 5/10
135/135 [==============================] - 14s 107ms/step - loss: 1.1451 - accuracy: 0.6898 - val_loss: 0.7365 - val_accuracy: 0.7954
Epoch 6/10
135/135 [==============================] - 15s 107ms/step - loss: 1.0957 - accuracy: 0.7016 - val_loss: 0.7053 - val_accuracy: 0.8056
Epoch 7/10
135/135 [==============================] - 14s 107ms/step - loss: 1.0099 - accur

In [27]:
# Giai đoạn 2: Fine-tune 50 layer cuối
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False  # Chỉ mở 50 lớp cuối

# Compile lại với learning rate nhỏ hơn
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), 
              loss='sparse_categorical_crossentropy', 
              metrics=['accuracy'])

In [28]:
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6)

In [29]:
print("==== GIAI ĐOẠN 2: Fine-tune 50 lớp cuối của InceptionV3 ====")
history2 = model.fit(datagen.flow(X_train, y_train, batch_size=32), 
                     epochs=40, 
                     validation_data=(X_test, y_test), 
                     callbacks=[lr_scheduler])

==== GIAI ĐOẠN 2: Fine-tune 50 lớp cuối của InceptionV3 ====
Epoch 1/40
135/135 [==============================] - 20s 116ms/step - loss: 0.9632 - accuracy: 0.7530 - val_loss: 0.5965 - val_accuracy: 0.8241 - lr: 1.0000e-04
Epoch 2/40
135/135 [==============================] - 15s 110ms/step - loss: 0.6910 - accuracy: 0.8074 - val_loss: 0.5560 - val_accuracy: 0.8389 - lr: 1.0000e-04
Epoch 3/40
135/135 [==============================] - 15s 110ms/step - loss: 0.5626 - accuracy: 0.8447 - val_loss: 0.5392 - val_accuracy: 0.8435 - lr: 1.0000e-04
Epoch 4/40
135/135 [==============================] - 15s 109ms/step - loss: 0.5286 - accuracy: 0.8472 - val_loss: 0.5203 - val_accuracy: 0.8546 - lr: 1.0000e-04
Epoch 5/40
135/135 [==============================] - 15s 113ms/step - loss: 0.4567 - accuracy: 0.8738 - val_loss: 0.5082 - val_accuracy: 0.8611 - lr: 1.0000e-04
Epoch 6/40
135/135 [==============================] - 15s 110ms/step - loss: 0.3782 - accuracy: 0.8956 - val_loss: 0.4978 - val_a

In [30]:
history = {
    'accuracy': history1.history['accuracy'] + history2.history['accuracy'],
    'val_accuracy': history1.history['val_accuracy'] + history2.history['val_accuracy'],
    'loss': history1.history['loss'] + history2.history['loss'],
    'val_loss': history1.history['val_loss'] + history2.history['val_loss'],
}

In [31]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1, len(history['accuracy'])+1)), y=history['accuracy'], mode='lines+markers', name='Training Accuracy'))
fig.add_trace(go.Scatter(x=list(range(1, len(history['val_accuracy'])+1)), y=history['val_accuracy'], mode='lines+markers', name='Validation Accuracy'))
fig.update_layout(title='Model Accuracy', xaxis_title='Epoch', yaxis_title='Accuracy', title_font_size=24, title_x=0.5, xaxis_title_font_size=18, yaxis_title_font_size=18, font=dict(family="Arial, sans-serif", size=14), template='plotly_dark')
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=list(range(1, len(history['loss'])+1)), y=history['loss'], mode='lines+markers', name='Training Loss'))
fig.add_trace(go.Scatter(x=list(range(1, len(history['val_loss'])+1)), y=history['val_loss'], mode='lines+markers', name='Validation Loss'))
fig.update_layout(title='Model Loss', xaxis_title='Epoch', yaxis_title='Loss', title_font_size=24, title_x=0.5, xaxis_title_font_size=18, yaxis_title_font_size=18, font=dict(family="Arial, sans-serif", size=14), template='plotly_dark')
fig.show()

In [32]:
# Evaluate model
y_pred = np.argmax(model.predict(X_test), axis=-1)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

34/34 [==============================] - 2s 34ms/step
                precision    recall  f1-score   support

      antelope       0.75      0.82      0.78        11
        badger       1.00      1.00      1.00        19
           bat       0.64      0.82      0.72        11
          bear       1.00      0.77      0.87        13
           bee       0.88      0.94      0.91        16
        beetle       1.00      0.91      0.95        11
         bison       0.92      0.92      0.92        12
          boar       0.88      0.88      0.88        16
     butterfly       0.79      0.92      0.85        12
           cat       0.85      0.85      0.85        13
   caterpillar       0.82      0.82      0.82        11
    chimpanzee       0.91      0.83      0.87        12
     cockroach       1.00      0.87      0.93        15
           cow       0.81      0.87      0.84        15
        coyote       0.62      0.73      0.67        11
          crab       1.00      1.00      1.00    